# MLA

In [ ]:
import torch
from torch import nn
from xxx import RoPEEmbedding # 假设的RoPE模块


class MultiHeadLatentAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, q_lora_rank, kv_lora_rank, qk_nope_head_dim, qk_rope_head_dim, v_head_dim,
                 max_seq_len, dropout=0.1):
        super().__init__()
        assert qk_rope_head_dim % 2 == 0, "qk_rope_head_dim must be even"

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.q_lora_rank = q_lora_rank
        self.kv_lora_rank = kv_lora_rank
        self.qk_nope_head_dim = qk_nope_head_dim  # 每个 Q/K 头中不应用 RoPE 的维度
        self.qk_rope_head_dim = qk_rope_head_dim  # 每个 Q/K 头中应用 RoPE 的维度
        self.qk_head_dim = qk_nope_head_dim + qk_rope_head_dim
        self.v_head_dim = v_head_dim
        self.scale = self.qk_head_dim ** -0.5
        self.max_seq_len = max_seq_len

        # Query 低秩压缩，先将输入压缩到较小的潜空间，再恢复为多头 Query
        self.q_a_proj = nn.Linear(hidden_dim, q_lora_rank)
        self.q_a_layernorm = nn.LayerNorm(q_lora_rank)
        self.q_b_proj = nn.Linear(q_lora_rank, num_heads * self.qk_head_dim)

        # KV 联合低秩压缩：同时生成压缩后的 KV 潜表示和携带位置信息的 Key
        self.kv_a_proj = nn.Linear(hidden_dim, kv_lora_rank + qk_rope_head_dim)
        self.kv_a_layernorm = nn.LayerNorm(kv_lora_rank)
        self.kv_b_proj = nn.Linear(kv_lora_rank, num_heads * (qk_nope_head_dim + v_head_dim))

        self.o_proj = nn.Linear(num_heads * v_head_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.rope = RoPEEmbedding(qk_rope_head_dim, max_seq_len)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        # 将 Query 从低秩潜表示恢复成多个注意力头
        Q = self.q_b_proj(self.q_a_layernorm(self.q_a_proj(x)))
        Q = Q.view(batch_size, seq_len, self.num_heads, self.qk_head_dim).transpose(1, 2)
        Q_nope = Q[..., :self.qk_nope_head_dim] # 与位置无关的部分
        Q_rope = Q[..., self.qk_nope_head_dim:] # 需要注入位置信息的部分

        # kv_a_proj 的输出由两部分组成：压缩后的 KV 潜在表示，以及共享的位置 Key
        compressed_kv, K_rope = torch.split(
            self.kv_a_proj(x), [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1
        )
        KV = self.kv_b_proj(self.kv_a_layernorm(compressed_kv))
        KV = KV.view(batch_size, seq_len, self.num_heads, self.qk_nope_head_dim + self.v_head_dim).transpose(1, 2)
        
        # K_nope 和 V 由 compressed_kv 恢复，并分别展开到各个注意力头
        K_nope = KV[..., :self.qk_nope_head_dim]
        V = KV[..., self.qk_nope_head_dim:]

        # 只对 Q、K 中较小的位置部分应用旋转位置编码
        # 普通 MHA 对完整 Q、K 应用 RoPE，而 MLA 保留不经过 RoPE 的部分
        Q_rope = self.rope(Q_rope)
        K_rope = self.rope(K_rope.unsqueeze(1))
        
        # K_rope 不区分注意力头，将同一份位置 Key 共享给所有头，进一步降低缓存开销
        K_rope = K_rope.expand(-1, self.num_heads, -1, -1)

        # 拼接内容部分和位置编码过部分，得到最终用于计算注意力分数的 Q、K
        Q = torch.cat([Q_nope, Q_rope], dim=-1)
        K = torch.cat([K_nope, K_rope], dim=-1)

        attn_scores = Q @ K.transpose(-2, -1) * self.scale
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask, float('-inf'))
        attn_scores = torch.softmax(attn_scores, dim=-1)
        attn_scores = self.dropout(attn_scores)

        output = (attn_scores @ V).transpose(1, 2).reshape(batch_size, seq_len, self.num_heads * self.v_head_dim)
        output = self.o_proj(output)
        return output, attn_scores
